<a href="https://colab.research.google.com/github/inhyuk78/Re-implementation-of-MOLI-By-Hossein-Sharifi-Noghabi-/blob/main/04_Drug_Response_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from itertools import combinations

## **Importing X_train, X_test, y_train, y_test data**

In [ ]:
data = torch.load('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/multi_omics.pt')

X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

## **Train/Test split of multi-omics tensor data**

In [ ]:
X_train = X_train.float()
y_train = y_train.view(-1).float()

X_test = X_test.float()
y_test = y_test.view(-1).float()

## **NN model architecture**

In [ ]:
class MOLIClassifier(nn.Module):
  def __init__(self, input_dim):
    super().__init__()

    # Layer receiving 768-features input
    self.feature = nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.3)
    )

    # Embedding layer (32-dimensional embedding vectors output by this layer used to mine triplets for triplet loss)
    self.embed = nn.Sequential(
        nn.Linear(512, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.3)
    )

    # Final classifier layer
    self.fc_out = nn.Linear(32, 1)     # Single logit output for BCELoss

  def forward(self,x):
    z = self.feature(x)
    embedding = self.embed(z)
    pred = torch.sigmoid(self.fc_out(embedding)).squeeze(1)
    return pred, embedding

## **Triplet Mining**

In [ ]:
def mine_all_triplets(embedding, labels):
  anchors, positives, negatives = [], [], []

  sensitive_indices = torch.where(labels==1)[0]
  resistant_indices = torch.where(labels==0)[0]

  # For sensitive (label=1): Anchor = Sensitive | Positive = Sensitive | Negative = Resistant
  for anchor_indices, positive_indices in combinations(sensitive_indices.tolist(), 2):   # pull unique pair combinations of anchor/positive
    for negative_indices in resistant_indices.tolist():  # pull single negative
      anchors.append(embedding[anchor_indices])
      positives.append(embedding[positive_indices])
      negatives.append(embedding[negative_indices])

  # For resistant (label=0): Anchor = Resistant | Positive = Resistant | Negative = Sensitive
  for anchor_indices, positive_indices in combinations(resistant_indices.tolist(), 2):
    for negative_indices in sensitive_indices.tolist():
      anchors.append(embedding[anchor_indices])
      positives.append(embedding[positive_indices])
      negatives.append(embedding[negative_indices])

  if len(anchors) == 0:
    return None, None, None

  return (
      torch.stack(anchors),
      torch.stack(positives),
      torch.stack(negatives)
  )

## **Combined Loss**

In [ ]:
# J = BCELoss + TripletLoss

bce_loss = nn.BCELoss()
triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

    # margin = specifies how much further away negative should be than positive
        # Eg) Distance between anchor - positive = 2
        # Eg) Distance between anchor - negative = 5
        # Eg) If margin=1: 2-5+1 = 0, thus loss = max(-2,0) = 0
          # Thus, no penalty to negative, meaning negative is already far enough away

        # Eg) If margin=5: 2-5+5 = 2, thus loss = max(2,0) = 2
          # Thus, penalized, meaning model will learn to place negative further away (or positive closer) during training


    # p = which distance metric used
        # p=2: Standard Euclidean distance

In [ ]:
def combined_loss(preds, labels, anchors, positives, negatives, gamma=0.1):
  # BCELoss
  pred_loss = bce_loss(preds, labels.float())

  # TripletLoss
  if anchors is None:
    trip_loss = torch.tensor(0.0, requires_grad=True)
  else:
    trip_loss = triplet_loss(anchors, positives, negatives)

  total_loss = pred_loss + (gamma * trip_loss)
  return total_loss, pred_loss, trip_loss

In [ ]:
dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
def train_model(model, dataloader, optimizer, epochs):
  model.train()

  for epoch in range(epochs):
    epoch_loss = 0
    epoch_pred_loss = 0
    epoch_trip_loss = 0

    for batch_x, batch_y in dataloader:
      optimizer.zero_grad()

      # Forward prop
      preds, embeddings = model(batch_x)

      # Mine anchors, positives, negatives for triplet loss
      anchors, positives, negatives = mine_all_triplets(embeddings, batch_y)

      # Combined loss
      total_loss, pred_loss, trip_loss = combined_loss(preds, batch_y, anchors, positives, negatives)

      # Backprop + Update
      total_loss.backward()
      optimizer.step()

      epoch_loss += total_loss.item()
      epoch_pred_loss += pred_loss.item()
      epoch_trip_loss += trip_loss.item()

    n_batches = len(dataloader)
    print(
        f'Epoch [{epoch+1}/{epochs}]'
        f'Total: {epoch_loss/n_batches}'
        f'BCE: {epoch_pred_loss/n_batches}'
        f'Triplet: {epoch_trip_loss/n_batches}'
    )

## **Training & Evaluating Model**

In [ ]:
def evaluate_model(model, dataloader):
  model.eval()

  all_preds = []
  all_labels = []

  with torch.no_grad():
    for batch_x, batch_y in dataloader:
      preds, _ = model(batch_x)
      all_preds.append(preds)
      all_labels.append(batch_y)

  all_preds = torch.cat(all_preds).numpy()   # Convert tensor -> numpy array
  all_labels = torch.cat(all_labels).numpy()

  auc = roc_auc_score(all_labels, all_preds)

  binary_preds = (all_preds >= 0.5).astype(int)
  accuracy = (binary_preds == all_labels).mean()
  return auc, accuracy

In [ ]:
# Function that trains one model from scratch
    # note: Adam optimizer stores parameters so important to start fresh per training model

def run_variant(train_fn, seed=42, epochs=20):
  torch.manual_seed(seed)
  model = MOLIClassifier(input_dim=768)
  optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
  train_fn(model, dataloader, optimizer, epochs=epochs)
  return evaluate_model(model,test_dataloader)

## **<u>Ablation Study</u>:**
## **How does Triplet loss (as in original paper) vs No Triplet loss vs Contrastive loss compare for model performance**

#### **Triplet loss:** Works on triplets. Pulls positive closer to anchor, pushes apart negative from anchor embedding space.
#### **Contrastive loss:** Works on pairs. Pulls similar pair, Pushes apart dissimilar pairs in embedding space.


#### 1) No Triplet Loss (pure classification loss)

In [ ]:
def train_model_no_trip_loss(model, dataloader, optimizer, epochs):
  model.train()

  for epoch in range(epochs):
    epoch_loss = 0
    epoch_pred_loss = 0

    for batch_x, batch_y in dataloader:
      optimizer.zero_grad()
      preds, _ = model(batch_x)
      pred_loss = bce_loss(preds, batch_y.float())
      pred_loss.backward()
      optimizer.step()

      epoch_loss += pred_loss.item()
      epoch_pred_loss += pred_loss.item()

    n_batches = len(dataloader)
    print(
        f'Epoch: [{epoch+1}/{epochs}]'
        f'Total: {epoch_loss/n_batches}'
        f'BCE: {epoch_pred_loss/n_batches}'
    )

#### 2) (Pairwise) Contrastive Loss

In [ ]:
class ContrastiveLoss(nn.Module):
  def __init__(self,margin=1.0):
    super().__init__()
    self.margin = margin

  def forward(self,emb1,emb2,label):
    # label: 1 if emb1, emb2 are 'similar' pair (same drug response), label:0 if emb1, emb2 are different

    distance = F.pairwise_distance(emb1, emb2)  # Finds Euclidean distance

    # If similar (label=1): loss = distance squared (= aim for small distance to bring similar pairs together)
    # If different (label=0): loss = max(0, self.margin-distance) squared (= aim for large distance to push apart different pairs)
        # note: relu, max(0,x), used to clip any negative value to zero since 'negative loss' is not possible
    loss = label * distance.pow(2) + (1 - label) * F.relu(self.margin - distance).pow(2)
    return loss.mean()

In [ ]:
def mine_all_pairs(embedding, labels):
  emb1_list, emb2_list, pair_labels = [], [], []

  sensitive_indices = torch.where(labels==1)[0]
  resistant_indices = torch.where(labels==0)[0]

  # Label: 1 (sensitive-sensitive pairs)
  for i, j in combinations(sensitive_indices.tolist(),2):
    emb1_list.append(embedding[i])
    emb2_list.append(embedding[j])
    pair_labels.append(1)

  # Label: 1 (resistant-resistant pairs)
  for i, j in combinations(resistant_indices.tolist(),2):
    emb1_list.append(embedding[i])
    emb2_list.append(embedding[j])
    pair_labels.append(1)

  # Label: 0 (sensitive-resistant pairs)
  for i in sensitive_indices.tolist():
    for j in resistant_indices.tolist():
      emb1_list.append(embedding[i])
      emb2_list.append(embedding[j])
      pair_labels.append(0)

  if len(emb1_list) == 0:
    return None, None, None

  return (
      torch.stack(emb1_list),
      torch.stack(emb2_list),
      torch.tensor(pair_labels, dtype=torch.float32, device=embedding.device)
  )

In [ ]:
contrastive_loss_fn = ContrastiveLoss(margin=1.0)

def combined_loss_contrastive(preds, labels, emb1, emb2, pair_labels, gamma=0.1):
  # BCELoss
  pred_loss = bce_loss(preds, labels.float())

  # Contrastive loss
  if emb1 is None:
    contras_loss = torch.tensor(0.0, requires_grad=True, device=preds.device)
  else:
    contras_loss = contrastive_loss_fn(emb1, emb2, pair_labels)

  total_loss = pred_loss + (gamma * contras_loss)
  return total_loss, pred_loss, contras_loss

In [ ]:
def train_model_contrastive_loss(model, dataloader, optimizer, epochs):
  model.train()

  for epoch in range(epochs):
    epoch_loss = 0
    epoch_pred_loss = 0
    epoch_contrastive_loss = 0

    for batch_x, batch_y in dataloader:
      optimizer.zero_grad()

      # Forward prop
      preds, embeddings = model(batch_x)

      # Mine emb1, emb2, pair_labels for contrastive loss
      emb1, emb2, pair_labels = mine_all_pairs(embeddings, batch_y)

      # Combined loss
      total_loss, pred_loss, contrastive_loss = combined_loss_contrastive(preds, batch_y, emb1, emb2, pair_labels)

      # Backprop + Update loss
      total_loss.backward()
      optimizer.step()

      epoch_loss += total_loss.item()
      epoch_pred_loss += pred_loss.item()
      epoch_contrastive_loss += contrastive_loss.item()

    n_batches = len(dataloader)
    print(
        f'Epoch [{epoch+1}/{epochs}]',
        f'Total: {epoch_loss/n_batches}',
        f'BCE: {epoch_pred_loss/n_batches}',
        f'Contrastive: {epoch_contrastive_loss/n_batches}'
    )

In [ ]:
auc_triplet, acc_triplet = run_variant(train_model)
auc_bce, acc_bce = run_variant(train_model_no_trip_loss)
auc_contrastive, acc_contrastive = run_variant(train_model_contrastive_loss)

print()
print('_________________________________________________________________________')
print(f'AUC, accuracy of model with triplet loss: {auc_triplet} | {acc_triplet}')
print('_________________________________________________________________________')
print(f'AUC, accuracy of model with NO triplet loss: {auc_bce} | {acc_bce}')
print('_________________________________________________________________________')
print(f'AUC, accuracy of model with contrastive loss: {auc_contrastive} | {acc_contrastive}')

Epoch [1/20]Total: 0.8011749386787415BCE: 0.6628908731720664Triplet: 1.3828405792062932
Epoch [2/20]Total: 0.717336735942147BCE: 0.603544598275965Triplet: 1.1379214308478616
Epoch [3/20]Total: 0.708464882590554BCE: 0.6040605469183489Triplet: 1.044043410908092
Epoch [4/20]Total: 0.6785959167913957BCE: 0.5752272443337874Triplet: 1.033686719157479
Epoch [5/20]Total: 0.667500376701355BCE: 0.5638078288598494Triplet: 1.0369255434383045
Epoch [6/20]Total: 0.6580280553210865BCE: 0.5555376356298273Triplet: 1.0249040831219067
Epoch [7/20]Total: 0.6409661065448414BCE: 0.5409615608778867Triplet: 1.0000455379486084
Epoch [8/20]Total: 0.6393821347843517BCE: 0.5409179844639518Triplet: 0.9846415302970193
Epoch [9/20]Total: 0.6311742771755565BCE: 0.5341549597003243Triplet: 0.9701930934732611
Epoch [10/20]Total: 0.6283712387084961BCE: 0.5281010974537242Triplet: 1.0027014829895713
Epoch [11/20]Total: 0.6023190075700934BCE: 0.5083789310672067Triplet: 0.939400775866075
Epoch [12/20]Total: 0.614679786291989

###**Multi-seed comparison**

In [ ]:
results = {
    'triplet': [],
    'no_triplet': [],
    'contrastive': []
}

In [ ]:
for seed in range(5):
  results['triplet'].append(run_variant(train_model, seed=seed))
  results['no_triplet'].append(run_variant(train_model_no_trip_loss, seed=seed))
  results['contrastive'].append(run_variant(train_model_contrastive_loss, seed=seed))

for variant, runs in results.items():
  aucs = []

  for r in runs:
    aucs.append(r[0])

  print()
  print(f'{variant}: AUC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}')

Epoch [1/20]Total: 0.8038113821636547BCE: 0.6590070399371061Triplet: 1.448043335567821
Epoch [2/20]Total: 0.719932805408131BCE: 0.5998475009744818Triplet: 1.2008529576388272
Epoch [3/20]Total: 0.6963048577308655BCE: 0.5807758380066265Triplet: 1.1552902243354104
Epoch [4/20]Total: 0.6776702349836176BCE: 0.5700038183819164Triplet: 1.0766641931100325
Epoch [5/20]Total: 0.6556849371303212BCE: 0.5527982115745544Triplet: 1.0288673530925403
Epoch [6/20]Total: 0.6420786109837618BCE: 0.5442795347083699Triplet: 0.9779907410795038
Epoch [7/20]Total: 0.6550612503832037BCE: 0.5487145781517029Triplet: 1.063466733152216
Epoch [8/20]Total: 0.6296132586219094BCE: 0.5325801778923381Triplet: 0.9703307531096719
Epoch [9/20]Total: 0.6328898072242737BCE: 0.5351947302168066Triplet: 0.9769508242607117
Epoch [10/20]Total: 0.6099538857286627BCE: 0.5131879150867462Triplet: 0.9676597226749767
Epoch [11/20]Total: 0.6071979999542236BCE: 0.5117998854680494Triplet: 0.9539811069315131
Epoch [12/20]Total: 0.60745526443